# Module 4: Convolutional Neural Networks (CNN)

## Learning Objectives
- Understand convolution operations and their parameters
- Learn about pooling layers and their purpose
- Explore popular CNN architectures
- Implement CNN for image classification

## 4.1 Introduction to CNNs

Convolutional Neural Networks are specifically designed for processing grid-like data, such as images. Key components:
- **Convolutional layers**: Apply filters to detect features
- **Pooling layers**: Reduce spatial dimensions
- **Fully connected layers**: Perform classification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10, mnist
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## 4.2 Convolution Operation

In [ ]:
def conv2d_manual(input_image, kernel, stride=1, padding=0):
    """Manual implementation of 2D convolution"""
    # Add padding
    if padding > 0:
        input_image = np.pad(input_image, padding, mode='constant')
    
    input_height, input_width = input_image.shape
    kernel_height, kernel_width = kernel.shape
    
    # Calculate output dimensions
    output_height = (input_height - kernel_height) // stride + 1
    output_width = (input_width - kernel_width) // stride + 1
    
    output = np.zeros((output_height, output_width))
    
    # Perform convolution
    for i in range(0, output_height):
        for j in range(0, output_width):
            h_start = i * stride
            h_end = h_start + kernel_height
            w_start = j * stride
            w_end = w_start + kernel_width
            
            output[i, j] = np.sum(input_image[h_start:h_end, w_start:w_end] * kernel)
    
    return output

# Create a simple image and kernel
image = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
])

# Edge detection kernel
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])

# Apply convolution
conv_result = conv2d_manual(image, kernel, stride=1, padding=0)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image, cmap='viridis')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(kernel, cmap='viridis')
axes[1].set_title('Kernel (Edge Detection)')
axes[1].axis('off')

axes[2].imshow(conv_result, cmap='viridis')
axes[2].set_title('Convolution Result')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4.3 Pooling Operations

In [ ]:
def max_pool2d(input_image, pool_size=2, stride=2):
    """Manual implementation of max pooling"""
    input_height, input_width = input_image.shape
    
    output_height = (input_height - pool_size) // stride + 1
    output_width = (input_width - pool_size) // stride + 1
    
    output = np.zeros((output_height, output_width))
    
    for i in range(0, output_height):
        for j in range(0, output_width):
            h_start = i * stride
            h_end = h_start + pool_size
            w_start = j * stride
            w_end = w_start + pool_size
            
            output[i, j] = np.max(input_image[h_start:h_end, w_start:w_end])
    
    return output

def avg_pool2d(input_image, pool_size=2, stride=2):
    """Manual implementation of average pooling"""
    input_height, input_width = input_image.shape
    
    output_height = (input_height - pool_size) // stride + 1
    output_width = (input_width - pool_size) // stride + 1
    
    output = np.zeros((output_height, output_width))
    
    for i in range(0, output_height):
        for j in range(0, output_width):
            h_start = i * stride
            h_end = h_start + pool_size
            w_start = j * stride
            w_end = w_start + pool_size
            
            output[i, j] = np.mean(input_image[h_start:h_end, w_start:w_end])
    
    return output

# Apply pooling to our convolution result
max_pooled = max_pool2d(conv_result, pool_size=2, stride=2)
avg_pooled = avg_pool2d(conv_result, pool_size=2, stride=2)

# Visualize pooling results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(conv_result, cmap='viridis')
axes[0].set_title('After Convolution')
axes[0].axis('off')

axes[1].imshow(max_pooled, cmap='viridis')
axes[1].set_title('Max Pooling')
axes[1].axis('off')

axes[2].imshow(avg_pooled, cmap='viridis')
axes[2].set_title('Average Pooling')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Original shape: {conv_result.shape}")
print(f"Max pooled shape: {max_pooled.shape}")
print(f"Avg pooled shape: {avg_pooled.shape}")

## 4.4 Building a CNN with TensorFlow/Keras

In [ ]:
# Load and preprocess MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape for CNN (add channel dimension)
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

# One-hot encode labels
y_train_onehot = keras.utils.to_categorical(y_train, 10)
y_test_onehot = keras.utils.to_categorical(y_test, 10)

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Training labels shape: {y_train_onehot.shape}")

# Visualize some samples
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.title(f'Label: {y_train[i]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build a simple CNN model
def build_cnn_model(input_shape=(28, 28, 1), num_classes=10):
    model = keras.Sequential([
        # First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second convolutional block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fully connected layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create and compile the model
model = build_cnn_model()
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## 4.5 Training the CNN

In [ ]:
# Set up callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
]

# Train the model
history = model.fit(
    x_train, y_train_onehot,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Plot loss
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4.6 Model Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test, y_test_onehot, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Make predictions
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 4.7 Visualizing Feature Maps

In [ ]:
# Create a model to extract feature maps
layer_outputs = [layer.output for layer in model.layers[:6]]  # First 6 layers
feature_map_model = keras.Model(inputs=model.input, outputs=layer_outputs)

# Get feature maps for a sample image
sample_image = x_test[0:1]  # Take first test image
feature_maps = feature_map_model.predict(sample_image)

# Visualize feature maps from different layers
layer_names = ['Conv2D_1', 'BatchNorm_1', 'Conv2D_2', 'BatchNorm_2', 'MaxPool2D_1', 'Dropout_1']

for layer_name, feature_map in zip(layer_names, feature_maps):
    if len(feature_map.shape) == 4:  # Check if it's a convolutional layer
        num_features = feature_map.shape[-1]
        size = feature_map.shape[1]
        
        # Display up to 16 feature maps
        display_features = min(16, num_features)
        
        fig, axes = plt.subplots(4, 4, figsize=(12, 12))
        fig.suptitle(f'Feature Maps - {layer_name}', fontsize=16)
        
        for i in range(display_features):
            row = i // 4
            col = i % 4
            
            if row < 4 and col < 4:
                axes[row, col].imshow(feature_map[0, :, :, i], cmap='viridis')
                axes[row, col].axis('off')
                axes[row, col].set_title(f'Filter {i+1}')
        
        # Hide empty subplots
        for i in range(display_features, 16):
            row = i // 4
            col = i % 4
            if row < 4 and col < 4:
                axes[row, col].axis('off')
        
        plt.tight_layout()
        plt.show()

# Show the original image
plt.figure(figsize=(5, 5))
plt.imshow(sample_image[0].reshape(28, 28), cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.show()

## 4.8 Popular CNN Architectures

### LeNet-5 (1998)
- One of the first successful CNNs
- 7 layers (excluding input)
- Used for handwritten digit recognition

### AlexNet (2012)
- Won ImageNet competition 2012
- Deeper network with 8 layers
- Introduced ReLU activation and dropout

### VGGNet (2014)
- Very deep (16-19 layers)
- Used only 3x3 convolution filters
- Simple and uniform architecture

### ResNet (2015)
- Introduced residual connections
- Enabled training of very deep networks (152+ layers)
- Solved vanishing gradient problem

### EfficientNet (2019)
- Balances network depth, width, and resolution
- State-of-the-art efficiency
- Compound scaling method

## 4.9 Key Takeaways

- **Convolution** applies filters to detect local patterns
- **Pooling** reduces spatial dimensions and provides translation invariance
- **CNNs** automatically learn hierarchical feature representations
- **Batch normalization** stabilizes training
- **Dropout** prevents overfitting
- **Deep architectures** like ResNet enable very deep networks

## Exercises

1. Experiment with different filter sizes (3x3, 5x5, 7x7)
2. Try different pooling strategies (max, average, global)
3. Implement data augmentation for better generalization
4. Build a deeper network and observe the effects
5. Transfer learning: Use pre-trained models on different datasets